# Anees — Arabic Handwriting Quality & Similarity Rating
### Unit 7 · Computer Vision · Final Project

A child copies an Arabic letter from a Naskh model. This notebook scores **how
accurately they copied it** — a continuous **0–100 rating**, not OCR. The letter
is known in advance; the question is how well it was written.

```
photo → page detection → perspective warp → ink extraction → ruled-line removal
      → segmentation → 64×64 normalisation → CNN regressor → calibrated rating
```

Everything lives in one class, `AneesHandwritingCV`, in `handwriting_cv.py`.

---

### Before you run

1. **Runtime → Change runtime type → T4 GPU → Save**
2. Upload **`handwriting_cv.py`** and **`hijja2.npz`** with the folder icon on
   the left (or run the setup cell below, which will ask for them).
3. **Runtime → Run all**

### How long it takes, on a T4

| Preset | What it trains | Roughly |
|---|---|---|
| `QUICK` | production only, 1 member, 20k pairs, 6 epochs | **~5 min** |
| `STANDARD` | scratch + production, 1 member, 90k pairs | **~35 min** |
| `FULL` | all three, 2-member ensemble | **~1.5 hours** |

Run `QUICK` first — it proves the whole path works end to end before you spend
an hour on it. Each model is **checkpointed to `artifacts/` the moment it
finishes**, so if Colab reclaims the runtime partway through a `FULL` run you
lose only the model that was training, not the ones already done.

## 0 · Setup

In [ ]:
# Colab already has TensorFlow, OpenCV, scikit-image and matplotlib.
# These two only matter for drawing Arabic text in figure titles.
!pip -q install arabic-reshaper python-bidi 2>/dev/null

import os, sys, glob
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

# --- make sure handwriting_cv.py is here -----------------------------------
if not os.path.exists("handwriting_cv.py"):
    hits = glob.glob("/content/**/handwriting_cv.py", recursive=True)
    if hits:
        os.replace(hits[0], "handwriting_cv.py")
    elif IN_COLAB:
        from google.colab import files
        print("upload handwriting_cv.py")
        files.upload()
assert os.path.exists("handwriting_cv.py"), "handwriting_cv.py is missing"

# --- and that Hijja2 is somewhere it can be found ---------------------------
def _have_data():
    return any(os.path.exists(p) for p in
               ("hijja2.npz", "database cv/hijja2.npz", "/content/hijja2.npz",
                "/content/drive/MyDrive/hijja2/hijja2.npz"))

if not _have_data() and IN_COLAB:
    print("upload hijja2.npz  (5.9 MB — the packed Hijja2 dataset)")
    from google.colab import files
    files.upload()

print("ready")

In [ ]:
from handwriting_cv import AneesHandwritingCV
import numpy as np, matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
#  Pick one.  Start with "QUICK" — it proves the whole path works in ~5 min.
# ---------------------------------------------------------------------------
PRESET = "QUICK"          # "QUICK" | "STANDARD" | "FULL"

PRESETS = {
    "QUICK":    dict(n_train_pairs=20000, n_test_pairs=5000,  epochs=6,
                     ft_epochs=3,  ensemble=1,
                     train_scratch=False, train_mobilenet=False),
    "STANDARD": dict(n_train_pairs=90000, n_test_pairs=15000, epochs=35,
                     ft_epochs=14, ensemble=1,
                     train_scratch=True,  train_mobilenet=False),
    "FULL":     dict(n_train_pairs=90000, n_test_pairs=15000, epochs=35,
                     ft_epochs=14, ensemble=2,
                     train_scratch=True,  train_mobilenet=True),
}

cv = AneesHandwritingCV(**PRESETS[PRESET])
print(f"preset: {PRESET}")
cv.device_report()        # <- must say GPU. If it says NONE, fix the runtime.

## 1 · The data

**Hijja2** — 47,434 Arabic characters handwritten by **591 Saudi school children
aged 7–12**, collected in Riyadh, Jan–Apr 2019. 32×32 grayscale, 29 classes
(28 letters + hamza), split 37,933 train / 9,501 test.

It was chosen over AHCD because it is the *same population as the users* —
children, not adults. A model trained on adult handwriting would mark a normal
seven-year-old's letter as bad.

Contributors: Najwa Altwaijry, Monera Al-Megren, Haya Al-Shumisi, Lamya
Al-Arwan, Isra Al-Turaiki (King Saud University).

In [ ]:
cv.load_dataset()
summary = cv.dataset_summary()

In [ ]:
cv.show_class_distribution();

In [ ]:
cv.show_dataset(n=40);

## 2 · The Amiri model — what the child is copying

**Amiri** is the classical Naskh face from Google Fonts. It is rendered as the
reference "model" in **all four contextual forms** per letter (isolated,
initial, medial, final), because Arabic is cursive: a child's initial `بـ` must
not be judged against an isolated `ب`.

Hijja2 records the letter but never the form, so `best_form()` recovers it at
scoring time by picking whichever form the writing overlaps best.

Both sides — the child's stroke and the model — go through **exactly the same**
normalisation: crop to the ink, scale to a fixed box, centre, skeletonise, then
re-inflate to 2 px. That last step matters: it means the network cannot score a
letter well just because the child pressed harder with the pen.

In [ ]:
cv.build_reference_bank()
cv.show_reference_bank(letters=list(range(1, 13)));

## 3 · Where the continuous label comes from

This is the hard question in the project. Hijja2 tells us *which* letter a child
wrote, never *how well*. Inventing a neatness number would just teach the
network our invention, so the labels come from three sources, each honest about
what it is:

| | Source | Label | Status |
|---|---|---|---|
| **A** | the Amiri model degraded by a measured amount `t` | `1 − t` | **ground truth** |
| **B** | a real child's letter vs its Amiri model | classical geometric score | **proxy** |
| **C** | a real child's letter vs a *different* letter's model | ≈ 0 | **ground truth** |

Strong results on **B** alone would only prove the network memorised the
geometric formula. Results on **A** and **C** show it learned to see distortion
and letter mismatch directly. **Every table below reports all three separately.**

In [ ]:
# Source A: the four errors children actually make, applied in a measured amount.
# The amount IS the label — ground truth by construction.
cv.show_degradations("ب");

In [ ]:
cv.build_pairs(split="train")
cv.build_pairs(split="test")

In [ ]:
cv.show_pairs(n=10);

In [ ]:
cv.label_histogram();

## 4 · What the network is shown

Each pair is stacked into **one image** so both models stay single-stream and
MobileNetV2's ImageNet weights remain usable:

* `R` = the child's stroke  `G` = the Amiri model  `B` = their overlap

The production model gets **three more channels**: the disagreement (XOR) and
the decayed distance transforms of both sides. That last pair is what the
classical Hausdorff term measures — handing the network the field directly lets
it *see* "this stroke is 9 px from where it belongs" instead of re-deriving the
idea from binary pixels.

In [ ]:
i = 3
ch  = cv.train_pairs["childs"][i]
ref = cv.ref_arr[cv.train_pairs["rcls"][i], cv.train_pairs["rfrm"][i]]
six = cv.assemble(ch, ref, channels=6)

names = ["0 child", "1 Amiri model", "2 overlap (AND)",
         "3 disagreement (XOR)", "4 distance to model", "5 distance to child"]
fig, axes = plt.subplots(1, 6, figsize=(16, 2.9))
for k, ax in enumerate(axes):
    ax.imshow(six[..., k], cmap="magma" if k >= 4 else "gray")
    ax.set_title(names[k], fontsize=9); ax.axis("off")
fig.suptitle(f"the six input channels   (label {cv.train_pairs['y'][i]:.2f}, "
             f"source {cv.train_pairs['kinds'][i]})", fontsize=12)
plt.tight_layout(); plt.show()

## 5 · The three models

| | Architecture | From |
|---|---|---|
| **A** `scratch` | CNN from scratch, `Conv→MaxPool` ×4 + dense, 1 sigmoid unit | unit CV_2 |
| **B** `mobilenet` | MobileNetV2, feature extraction → fine-tune the top 60 layers at 10× lower LR | unit CV_3 |
| **C** `production` | residual CNN, 6-channel input, joint-affine augmentation, cosine schedule, 2-seed ensemble | the accuracy answer |

A and B exist so the two architectures from the unit can be compared honestly on
identical data. **C is the one the app ships**, and it is where the accuracy
comes from — five things, each of which moves the error down:

1. **Six input channels**, so the geometry is visible rather than inferred.
2. **Residual blocks with BatchNorm** — depth without the gradient dying.
3. **Average *and* max pooling concatenated.** Average pooling reports how wrong
   the letter is overall; max pooling reports the single worst place. A child's
   letter is usually mostly right with one bad stroke, and averaging alone hides
   exactly that.
4. **Label-preserving augmentation.** One small affine transform applied to the
   child and the model *together*. Rotating both by 3° does not make the copy
   any better or worse, so the network is pushed to measure the two against each
   other instead of against the canvas.
5. **A 2-seed ensemble + test-time augmentation.** The runs make different
   mistakes and the mistakes cancel.

In [ ]:
# The long cell. Every model is written to artifacts/ as soon as it finishes,
# so a dropped runtime does not cost you the ones already trained.
cv.train()

## 6 · Results

`MAE` is the number to quote: it is in the same units as the rating the child
sees, so an MAE of 3 means the rating is typically within 3 points of the label.

The **A / B / C** columns are the honest part. `geometric` is the classical
scorer, included as the baseline the CNNs have to beat.

In [ ]:
cv.evaluate();

In [ ]:
cv.show_history();

In [ ]:
cv.show_scatter('production');

## 7 · Calibration — what "82%" is allowed to mean

Raw similarity against typeset Amiri is a harsh scale: real children land in the
0.2–0.5 band, because **no seven-year-old writes like a font**. Showing a child
"34%" would be both discouraging and meaningless.

So the raw score is mapped through the distribution of *real children's* scores,
and the number the child sees means:

> **"neater than N% of children aged 7–12"**

which is exactly what 47,434 real samples entitle us to say.

In [ ]:
cv.calibrate()
cv.show_calibration();

## 8 · Scoring a real photo, end to end

The classical front end does five things before the network ever runs, none of
them learned:

1. find the sheet of paper and warp the camera perspective flat
2. pull out the pen strokes despite shadow, white balance and paper tint
3. delete the printed ruling **without** eating the strokes that cross it
4. isolate the writing — an Arabic letter is not one connected component, so the
   dots of `ب` / `ت` / `ن` have to be region-grown back in
5. normalise to the same 64×64 canvas a Hijja2 row becomes

In [ ]:
# A synthetic "phone photo" so the demo runs with nothing uploaded:
# Amiri warped into a wobbly hand, on ruled paper, tilted on a desk, lit
# unevenly and photographed at an angle.
photo = cv.make_sample("ب", paper="ruled", sloppiness=1.2, seed=11)

plt.figure(figsize=(7, 5))
plt.imshow(photo[:, :, ::-1]); plt.axis("off")
plt.title("the input: a photo of a child's notebook"); plt.show()

cv.show_pipeline(photo, "ب");

In [ ]:
result = cv.rate(photo, "ب")
cv.show_rating(result)

In [ ]:
import json
print(json.dumps({k: v for k, v in result.items() if not k.startswith("_")},
                 ensure_ascii=False, indent=1))

### Your own photo

Run the cell below, pick a photo, and change the letter to whichever one the
child was asked to write.

One letter or one word per photo, page filling the frame, phone held upright,
no hard shadow across the writing.

In [ ]:
cv.upload_and_rate("ب")      # <- change the letter

## 9 · Save everything

In [ ]:
cv.save()
cv.export_zip()     # in Colab this downloads automatically

## 10 · Honest limitations

* **No human ever graded a sample.** Source **B**'s labels are a geometric
  formula, so on that slice the network distils a metric rather than learning a
  new judgement of neatness. **The fix is small and concrete: have one teacher
  grade 300–500 real samples 1–5, then fine-tune the last dense layer.**
  Everything else is already built for it — only the label source changes.
* The comparison is **static** — it never sees stroke order, direction or speed.
  A letter drawn backwards but shaped correctly scores well.
* Hijja2 is 32×32; upsampling adds no detail, so fine differences in a curve are
  simply not in the data.
* One rendering of Amiri is the reference. A school teaching a slightly
  different Naskh model would lose points on letterforms that are not mistakes
  — swap the TTF to fix.
* Harakat are excluded from Hijja2, so the model has never seen one.

---

### Credits

Hijja2 dataset — Altwaijry, Al-Megren, Al-Shumisi, Al-Arwan, Al-Turaiki, King
Saud University. Amiri font — SIL Open Font License.